# EFG point-charge

Here we reproduce results discussed [here](https://github.com/apdioguardi/EFG_point_charge_lattice_sum/tree/main/YBa2Cu3O6).

In [1]:
import numpy as np
from ase import Atoms
from ase.io import read
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT)) 

from pcefg.src.pcefg import constants
from pcefg.src.pcefg.lattice import get_atom_kinds
from pcefg.src.pcefg.point_charge import PointChargeEFG
from pcefg.src.pcefg.point_charge import (
    compute_efg,
    diagonalize_EFG,
    point_charge_EFG,
    sphere_radius_convergence
)

In [3]:
import os
dpath='./'

In [4]:
file = "Cava_1990_EntryWithCollCode66903.cif" 
file = os.path.join(dpath, file)
st = Structure.from_file(file)
print(st)
# atoms = AseAtomsAdaptor.get_atoms(st) # ISSUE: AseAtomsAdaptor discard distinct atoms
atoms = read(file)

Full Formula (Ba2 Y1 Cu3 O6)
Reduced Formula: Ba2Y(CuO2)3
abc   :   3.854400   3.854400  11.817500
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (12)
  #  SP         a    b       c
---  -------  ---  ---  ------
  0  Ba2+     0.5  0.5  0.8056
  1  Ba2+     0.5  0.5  0.1944
  2  Y3+      0.5  0.5  0.5
  3  Cu1.67+  0    0    0
  4  Cu1.67+  0    0    0.6398
  5  Cu1.67+  0    0    0.3602
  6  O2-      0    0    0.8489
  7  O2-      0    0    0.1511
  8  O2-      0    0.5  0.6209
  9  O2-      0.5  0    0.6209
 10  O2-      0    0.5  0.3791
 11  O2-      0.5  0    0.3791


In [5]:
gamma_sternheimer = {  # or gamma_inf, anti-shielding factors, for the ions in your material
    'Na': -4.0,
    'O':  -2.2,
    'Cu': -5.5,
    # 'Mn': ...,  # add your own value here if you have one
}

# Formal ionic charges [units of e], only needed if compute_lattice_efg=True.
formal_charges = {
    'Na': +1, 'F': -1, 'Ca': +2, 'Li': +1, 'Cl': -1, 'Mg': +2, 'O': -2,
    'Al': +3,
    # add whatever your material contains
}

There are 2 distinct Cu (Cu1, Cu2) atoms with different charges.

There are 2 distinct O (O1, O2) atoms with same charges.

In [6]:
# Identify atom kinds

atm_kinds = get_atom_kinds(atoms)

print("\nAtom kinds:")
for kind, indices in atm_kinds.items():
    print(f"{kind}: {indices}")


print("\nCu1 indices:")
print(atm_kinds["Cu1"])
print("\nCu2 indices:")
print(atm_kinds["Cu2"])


Atom kinds:
Ba: [0, 1]
Y: [2]
Cu1: [3]
Cu2: [4, 5]
O1: [6, 7]
O2: [8, 9, 10, 11]

Cu1 indices:
[3]

Cu2 indices:
[4, 5]


In [7]:
Cu_nuclear_spin   = 3/2
Cu_Quadrupole_moment = -((0.2020+0.204)/2)*1e-28  # m^2 (avg of 63Cu & 65Cu with 69.17% & 30.83% abundance)
Cu_Quadrupole_moment  = -0.211e-28 # 63Cu

YBCO_charges = {'Y': +3, 'Ba': +2, 'Cu1': +1, 'Cu2': +2, 'O': -2}
sphere_radius=150
gamma_sternheimer=-5.5

In [8]:
probe_idx = atm_kinds["Cu1"][0]
probe_pos = atoms.get_scaled_positions()[probe_idx]  

res = compute_efg(
    atoms, 
    probe_position=probe_pos, 
    atomic_charges=YBCO_charges, 
    sphere_radius=sphere_radius,
    gamma_sternheimer=gamma_sternheimer, 
    exclude_indices=(), 
    extra_charges=None,
    coords_are_cartesian=False, # True or False, depending on 'probe_position' type, frac. or cart.
    nuclear_spin=Cu_nuclear_spin,
    quadrupole_moment=Cu_Quadrupole_moment,
    verbose=True
)

Point-charge EFG: summing 966178 charges within radius 150.000 Å.

EFG analysis for probe site at frac coord. (0.0000, 0.0000, 0.0000)
Vzz          = -1.25879309e+22 V/m^2
Vyy          =  6.29396547e+21 V/m^2
Vxx          =  6.29396547e+21 V/m^2
eta          =  0.00000000 (unitless)
chi_Q_MHz    =  64.22308617 MHz
nu_z_MHz     =  32.11154308 MHz
nu_Q_MHz     =  32.11154308 MHz

EFG tensor V_ab (V/m^2) =
----------------------------------------------------------------------
 [  6.29396547e+21, -4.06310094e+05, -2.92751445e+04 ]
 [ -3.13536784e+05,  6.29396547e+21,  6.62051406e+03 ]
 [ -8.18527228e+04, -5.68716944e+04, -1.25879309e+22 ]
----------------------------------------------------------------------
Trace(V_ab) = -1.14714e+09
Symmetric   = False

principal axes (unitless) = 
----------------------------------------------------------------------
 [  1.00000000e+00, -0.00000000e+00,  0.00000000e+00 ]
 [  0.00000000e+00,  1.00000000e+00,  0.00000000e+00 ]
 [  0.00000000e+00, -5.55111

remove EFG tensor numerical noises

In [9]:
res.keys()
EFG_latt = res['EFG_tensor']
print('BEFORE:')
print(f"Trace(V_ab) = {np.trace(EFG_latt): .5e}")
print(f"Symmetric   = {np.allclose(EFG_latt, EFG_latt.T)}")

efg_noise_threshold=1e-8

max_element = np.max(np.abs(EFG_latt))
if max_element > 0:
    EFG_latt[np.abs(EFG_latt) < efg_noise_threshold * max_element] = 0.0

print('AFTER:')
print(f"Trace(V_ab) = {np.trace(EFG_latt): .5e}")
print(f"Symmetric   = {np.allclose(EFG_latt, EFG_latt.T)}")

BEFORE:
Trace(V_ab) = -1.14714e+09
Symmetric   = False
AFTER:
Trace(V_ab) = -1.14714e+09
Symmetric   = True


or with `PointChargeEFG` calculators

In [10]:
probe_idx = atm_kinds["Cu1"][0]
probe_pos = atoms.get_scaled_positions()[probe_idx] 

PC = PointChargeEFG(
    atoms=atoms,
    charges=YBCO_charges,
    sphere_radius=sphere_radius,
    gamma_sternheimer=gamma_sternheimer
)

efg_tensor = PC.get_raw_tensor(
    position=probe_pos,
    coords_are_cartesian=False,
)

In [11]:
print('BEFORE:')
print(f"Trace(V_ab) = {np.trace(efg_tensor): .5e}")
print(f"Symmetric   = {np.allclose(efg_tensor, efg_tensor.T)}")

efg_noise_threshold=1e-8

max_element = np.max(np.abs(efg_tensor))
if max_element > 0:
    efg_tensor[np.abs(efg_tensor) < efg_noise_threshold * max_element] = 0.0

print('AFTER:')
print(f"Trace(V_ab) = {np.trace(efg_tensor): .5e}")
print(f"Symmetric   = {np.allclose(efg_tensor, efg_tensor.T)}")

BEFORE:
Trace(V_ab) = -1.14714e+09
Symmetric   = False
AFTER:
Trace(V_ab) = -1.14714e+09
Symmetric   = True


In [12]:
results = PC.compute_at(
    position=probe_pos,
    coords_are_cartesian=False,
    nuclear_spin=Cu_nuclear_spin,
    quadrupole_moment=Cu_Quadrupole_moment,
    verbose=True,
)

results

Point-charge EFG: summing 966178 charges within radius 150.000 Å.

EFG analysis for probe site at frac coord. (0.0000, 0.0000, 0.0000)
Vzz          = -1.25879309e+22 V/m^2
Vyy          =  6.29396547e+21 V/m^2
Vxx          =  6.29396547e+21 V/m^2
eta          =  0.00000000 (unitless)
chi_Q_MHz    =  64.22308617 MHz
nu_z_MHz     =  32.11154308 MHz
nu_Q_MHz     =  32.11154308 MHz

EFG tensor V_ab (V/m^2) =
----------------------------------------------------------------------
 [  6.29396547e+21, -4.06310094e+05, -2.92751445e+04 ]
 [ -3.13536784e+05,  6.29396547e+21,  6.62051406e+03 ]
 [ -8.18527228e+04, -5.68716944e+04, -1.25879309e+22 ]
----------------------------------------------------------------------
Trace(V_ab) = -1.14714e+09
Symmetric   = False

principal axes (unitless) = 
----------------------------------------------------------------------
 [  1.00000000e+00, -0.00000000e+00,  0.00000000e+00 ]
 [  0.00000000e+00,  1.00000000e+00,  0.00000000e+00 ]
 [  0.00000000e+00, -5.55111

{'Vxx': np.float64(6.293965470608317e+21),
 'Vyy': np.float64(6.29396547060835e+21),
 'Vzz': np.float64(-1.2587930941217812e+22),
 'eta': 2.665603438459426e-15,
 'V_aa': array([ 6.29396547e+21,  6.29396547e+21, -1.25879309e+22]),
 'nu_z_MHz': np.float64(32.11154308375103),
 'nu_Q_MHz': np.float64(32.11154308375103),
 'chi_Q_MHz': 64.22308616750206,
 'EFG_tensor': array([[ 6.29396547e+21, -4.06310094e+05, -2.92751445e+04],
        [-3.13536784e+05,  6.29396547e+21,  6.62051406e+03],
        [-8.18527228e+04, -5.68716944e+04, -1.25879309e+22]]),
 'principal_axes': array([[ 1.00000000e+00, -0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  1.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00, -5.55111512e-17,  1.00000000e+00]]),
 'probe_index': None,
 'probe_symbol': None,
 'probe_position': array([0., 0., 0.])}